In [1]:
%load_ext sql

In [25]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
spark=SparkSession.builder.appName("MyApp").getOrCreate()
data = [
    (1, "Alice", 29,"HR",40000,"23rdMarch2021"),
    (2, "Bob", 31,"Engineering",50000,"15thApril2021"),
    (3, "Cathy", 25,"Marketing",45000,"10thMay2021"),
    (4, "David", 28,"Finance",60000,"5thJune2021"),
    (5, "Eva", 35,"HR",70000,"20thJuly2021"),
    (6, "Frank", 30,"Engineering",55000,"12thAugust2021"),
    (7, "Grace", 27,"Marketing",48000,"18thSeptember2021"),
    (8, "Hannah", 32,"Finance",62000,"25thOctober2021"),
    (9, "Ian", 29,"HR",41000,"30thNovember2021"),
    (10, "Jack", 34,"Engineering",53000,"15thDecember2021"),
    (11, "Kathy", 26,"Marketing",47000,"1stJanuary2022"),
    (12, "Leo", 33,"Finance",61000,"10thFebruary2022"),
    (13, "Mia", 28,"HR",42000,"20thMarch2022"),
    (14, "Nina", 31,"Engineering",54000,"5thApril2022"),
    (15, "Oscar", 29,"Marketing",46000,"15thMay2022"),
    (16, "Paul", 27,"Finance",59000,"25thJune2022"),
    (17, "Quinn", 30,"HR",43000,"10thJuly2022"),
    (18, "Rachel", 32,"Engineering",56000,"20thAugust2022"),
    (19, "Sam", 28,"Marketing",49000,"30thSeptember2022"),
    (20, "Tina", 35,"Finance",63000,"15thOctober2022")
]
columns = ["id", "name", "age", "department", "salary", "joining_date"]
df = spark.createDataFrame(data, columns)   
df_cleaned=df.withColumn("joining_date_cleaned",
    F.coalesce(
        F.try_to_date(F.col("joining_date"),"d'st'MMMMyyyy"),
        F.try_to_date(F.col("joining_date"),"d'nd'MMMMyyyy"),
        F.try_to_date(F.col("joining_date"),"d'rd'MMMMyyyy"),
        F.try_to_date(F.col("joining_date"),"d'th'MMMMyyyy")
    )
)

In [26]:
df_cleaned.select("name", "joining_date", "joining_date_cleaned").show(5)

+-----+-------------+--------------------+
| name| joining_date|joining_date_cleaned|
+-----+-------------+--------------------+
|Alice|23rdMarch2021|          2021-03-23|
|  Bob|15thApril2021|          2021-04-15|
|Cathy|  10thMay2021|          2021-05-10|
|David|  5thJune2021|          2021-06-05|
|  Eva| 20thJuly2021|          2021-07-20|
+-----+-------------+--------------------+
only showing top 5 rows


In [29]:
df_cleaned.select("id","name","age","department","salary","joining_date_cleaned").show()

+---+------+---+-----------+------+--------------------+
| id|  name|age| department|salary|joining_date_cleaned|
+---+------+---+-----------+------+--------------------+
|  1| Alice| 29|         HR| 40000|          2021-03-23|
|  2|   Bob| 31|Engineering| 50000|          2021-04-15|
|  3| Cathy| 25|  Marketing| 45000|          2021-05-10|
|  4| David| 28|    Finance| 60000|          2021-06-05|
|  5|   Eva| 35|         HR| 70000|          2021-07-20|
|  6| Frank| 30|Engineering| 55000|          2021-08-12|
|  7| Grace| 27|  Marketing| 48000|          2021-09-18|
|  8|Hannah| 32|    Finance| 62000|          2021-10-25|
|  9|   Ian| 29|         HR| 41000|          2021-11-30|
| 10|  Jack| 34|Engineering| 53000|          2021-12-15|
| 11| Kathy| 26|  Marketing| 47000|          2022-01-01|
| 12|   Leo| 33|    Finance| 61000|          2022-02-10|
| 13|   Mia| 28|         HR| 42000|          2022-03-20|
| 14|  Nina| 31|Engineering| 54000|          2022-04-05|
| 15| Oscar| 29|  Marketing| 46

Find 2nd Highest salary in each department

In [ ]:
from pyspark.sql.window import Window

WindowSal=Window.partitionBy("department").orderBy(F.desc("salary"))
df_ranked=df_cleaned.withColumn("ranked_val",F.row_number().over(WindowSal))
